In [ ]:
!pip install emoji
!pip install wordcloud
!pip install nltk
import nltk
!pip install plotly
nltk.download('vader_lexicon')
import re
import pandas as pd
import numpy as np
import emoji
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from nltk.sentiment.vader import SentimentIntensityAnalyzer

def date_time(s):
    pattern = r'^(\d{1,2}\/\d{1,2}\/\d{2,4}), (\d{1,2}:\d{2})\s?([aA][mM]|[pP][mM]|[aA]\.[mM]\.|[pP]\.[mM]\.)? -'
    result = re.match(pattern, s)
    return bool(result)

def getDatapoint(line):
    splitline = line.split(' - ', 1)
    dateTime = splitline[0]
    date, time = dateTime.split(", ", 1)
    message = splitline[1]
    
    if ": " in message:
        splitmessage = message.split(": ", 1)
        author = splitmessage[0]
        message = splitmessage[1]
    else:
        author = None
        
    return date, time, author, message

data_list = []
conversation = r"WhatsApp Chat.txt"  # Copy the path of your Text File

with open(conversation, encoding="utf-8") as fp:
    fp.readline()
    messageBuffer = []
    date, time, author = None, None, None
    
    while True:
        line = fp.readline()
        if not line:
            break
        line = line.strip()
        
        if date_time(line):
            if len(messageBuffer) > 0:
                data_list.append([date, time, author, ' '.join(messageBuffer)])
            messageBuffer.clear()
            date, time, author, message = getDatapoint(line)
            messageBuffer.append(message)
        else:
            messageBuffer.append(line)
            
    if len(messageBuffer) > 0:
        data_list.append([date, time, author, ' '.join(messageBuffer)])

df = pd.DataFrame(data_list, columns=["Date", "Time", "Author", "Message"])
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y', errors='coerce')

data = df.dropna().copy()

sentiments = SentimentIntensityAnalyzer()
data["Positive"] = [sentiments.polarity_scores(i)["pos"] for i in data["Message"]]
data["Negative"] = [sentiments.polarity_scores(i)["neg"] for i in data["Message"]]
data["Neutral"] = [sentiments.polarity_scores(i)["neu"] for i in data["Message"]]

print(data)
x = sum(data["Positive"])
y = sum(data["Negative"])
z = sum(data["Neutral"])

def sentiment_score(a, b, c):
    if (a>b) and (a>c):
        print("Positive 😊")
    elif (b>a) and (b>c):
        print("Negative 😠")
    else:
        print("Neutral 🙂")
sentiment_score(x, y, z)

import re
import pandas as pd
import numpy as np
import emoji
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator

def date_time(s):
    s = s.replace('\u200e', '')
    pattern = r'^([0-9]{1,2}\/[0-9]{1,2}\/[0-9]{2,4}), ([0-9]{1,2}:[0-9]{2}(?::[0-9]{2})?)\s?(AM|PM|am|pm)? -'
    result = re.match(pattern, s)
    return bool(result)

def getDatapoint(line):
    line = line.replace('\u200e', '') 
    splitline = line.split(' - ', 1) 
    dateTime = splitline[0]
    
    if ", " in dateTime:
        date, time = dateTime.split(", ", 1)
    else:
        date, time = dateTime, None
        
    message = splitline[1] if len(splitline) > 1 else ""
    
    if ": " in message:
        splitmessage = message.split(": ", 1)
        author = splitmessage[0]
        message = splitmessage[1]
    else:
        author = None
        
    return date, time, author, message

data = []
conversation = r"WhatsApp Chat.txt"  # Copy the path of your Text File

with open(conversation, encoding="utf-8") as fp:
    messageBuffer = []
    date, time, author = None, None, None
    
    for line in fp:
        line = line.strip()
        if not line:
            continue
            
        if date_time(line):
            if len(messageBuffer) > 0:
                data.append([date, time, author, ' '.join(messageBuffer)])
            messageBuffer.clear()
            date, time, author, message = getDatapoint(line)
            messageBuffer.append(message)
        else:
            messageBuffer.append(line)
            
    if len(messageBuffer) > 0:
        data.append([date, time, author, ' '.join(messageBuffer)])

df = pd.DataFrame(data, columns=["Date", 'Time', 'Author', 'Message'])
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y', errors='coerce')
print(df.tail(20))
print(df.info())
print(df.Author.unique())

total_messages = df.shape[0]
print(total_messages)

media_messages = df[df["Message"]=='<Media omitted>'].shape[0]
print(media_messages)

def split_count(text):
    if not isinstance(text, str):
        return []
    return [e['emoji'] for e in emoji.emoji_list(text)]
df['emoji'] = df["Message"].apply(split_count)
emojis = sum(df['emoji'].str.len())
print(f"Total emojis used: {emojis}")

URLPATTERN = r'(https?://\S+)'
df['urlcount'] = df.Message.apply(lambda x: regex.findall(URLPATTERN, x)).str.len()
links = np.sum(df.urlcount)

print("Chats between User1 and User2")
print("Total Messages: ", total_messages)
print("Number of Media Shared: ", media_messages)
print("Number of Emojis Shared", emojis)
print("Number of Links Shared", links)

media_messages_df = df[df['Message'] == '<Media omitted>']
messages_df = df.drop(media_messages_df.index)
messages_df['Letter_Count'] = messages_df['Message'].apply(lambda s : len(s))
messages_df['Word_Count'] = messages_df['Message'].apply(lambda s : len(s.split(' ')))
messages_df["MessageCount"]=1

l = ["User1", "User2"]
for i in range(len(l)):
  req_df= messages_df[messages_df["Author"] == l[i]]
  print(f'Stats of {l[i]} -')
  print('Messages Sent', req_df.shape[0])
  words_per_message = (np.sum(req_df['Word_Count']))/req_df.shape[0]
  print('Average Words per message', words_per_message)
  media = media_messages_df[media_messages_df['Author'] == l[i]].shape[0]
  print('Media Messages Sent', media)
  emojis = sum(req_df['emoji'].str.len())
  print('Emojis Sent', emojis)
  links = sum(req_df["urlcount"])   
  print('Links Sent', links)

import plotly.express as px
fig = px.bar(
    top_emojis_df, 
    x='emoji', 
    y='count', 
    title='Top 15 Most Used Emojis',
    text_auto=True, 
    color='count', 
)
fig.update_layout(xaxis=dict(tickfont=dict(size=18)))
fig.show(renderer="notebook")

l = ["User1", "User2"]
for i in range(len(l)):
  dummy_df = messages_df[messages_df['Author'] == l[i]]
  text = " ".join(review for review in dummy_df.Message)
  stopwords = set(STOPWORDS)
  print('Author name',l[i])
  wordcloud = WordCloud(stopwords=stopwords, background_color="white").generate(text)
  plt.figure( figsize=(10,5))
  plt.imshow(wordcloud, interpolation='bilinear')
  plt.axis("off")
  plt.show()
